# From Speech to Print

## Google Colab

In [ ]:
# Si estas corriendo este notebook en Google Colab, descomentá este código:
#
# !git clone https://github.com/martindsq/from-speech.git
# %cd from-speech/notebooks

## Setup

Agregamos la carpeta padre al Path para poder importar el módulo `cowaver`:

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import cowaver

Definimos una función que nos será util para escuchar audios.

In [ ]:
import io
import os
import tempfile
import torch
import torchaudio
import ipywidgets as widgets
from cowaver.utils import AUDIO_SAMPLE_RATE

def audio_widget(waveform: torch.Tensor):
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)

    waveform = waveform.detach().cpu()

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as temp_file:
        temp_path = temp_file.name

    try:
        torchaudio.save(
            temp_path,
            waveform,
            AUDIO_SAMPLE_RATE,
        )

        with open(temp_path, "rb") as audio_file:
            audio_bytes = audio_file.read()
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    return widgets.Audio(
        value=audio_bytes,
        format="wav",
        autoplay=False,
        loop=False,
        controls=True,
    )

Configuramos las constantes de este notebook:

In [ ]:
from pathlib import Path
from cowaver.utils import encontrar_dispositivo

# Idioma, puede ser "french" o "spanish"
LANGUAGE = "french"

# Tamaño del dataset disponible
DATASET_SIZE = 480

# URL al checkpoint de CORnet-Z
CORNET_CKPT_URL = "https://s3.amazonaws.com/cornet-models/cornet_z-5c427c9c.pth"

# Tamaño del bottelenck del autoencoder
H_DIM = 128

# Número de filtros en el autoencoder
N_FILTERS = 16

# Rutas a los datos
workspace_path = Path("../") / LANGUAGE
compressed_phones_path = workspace_path / f"kalulu-phones-{DATASET_SIZE}.tar.xz"
compressed_spoken_path = workspace_path / f"kalulu-spoken-{DATASET_SIZE}.tar.xz"

print(f"Buscando {compressed_phones_path}", end="... ")
print("OK" if compressed_phones_path.exists() else "ERROR")

print(f"Buscando {compressed_spoken_path}", end="... ")
print("OK" if compressed_spoken_path.exists() else "ERROR")

# Carpeta donde se descomprimirán los datos
data_path = workspace_path / "data"

# Carpeta donde se guardan los checkpoints
checkpoints_path = workspace_path / "checkpoints" / str(DATASET_SIZE)

## Conjuntos de datos: TinyPairedMel

El conjunto de datos se divide en _Training Set_ y _Generalization Set_. El primero contiene un número de palabras con las que se entrenará la conciencia fonológica, mientras que el segundo contendrá palabras no vistas (pero sí por el Speech Autoencoder), que servirá para poner a prueba la capacidad de generalización de la conciencia fonológica.

Ejecutá la celda siguiente para descomprimir los datos y armar los conjuntos de _Training_ y _Generalization_.

In [ ]:
from cowaver.utils import descomprimir_archivo, listar_clases, separar_clases
from cowaver.datamodules import TinyPairedMel

phones_path = descomprimir_archivo(compressed_phones_path, data_path)
spoken_path = descomprimir_archivo(compressed_spoken_path, data_path)

palabras = listar_clases(phones_path / "train")

print("Total de palabras:", len(palabras))

palabras_a_entrenar, palabras_a_generalizar = separar_clases(palabras, fraction=0.2)
print("Palabras a entrenar:", len(palabras_a_entrenar))
print(f"Palabras a generalizar ({len(palabras_a_generalizar)}):", palabras_a_generalizar)

full_data = TinyPairedMel(phones_path, spoken_path, mel_bins=80, classes=palabras)
training_data = TinyPairedMel(phones_path, spoken_path, mel_bins=80, classes=palabras_a_entrenar)
generalization_data = TinyPairedMel(phones_path, spoken_path, mel_bins=80, classes=palabras_a_generalizar)

La siguiente celda permite explorar el conjunto de datos:

In [ ]:
import matplotlib.pyplot as plt
from cowaver.utils import extraer_waveform
from cowaver.plots import graficar_waveform, graficar_mel

exploration_set = full_data.test_set

@widgets.interact(idx=(0, len(exploration_set) - 1))
def explore_test_set(idx=0):
    (image, phonetized_mel, spoken_mel), target = exploration_set[idx]
    phonetized_waveform = extraer_waveform(phonetized_mel)
    spoken_waveform = extraer_waveform(spoken_mel)

    out = widgets.Output()
    with out:
        fig, axes = plt.subplots(2, 2, figsize=(10, 4), layout="constrained")
        fig.suptitle(exploration_set.classes[target])
        graficar_waveform(phonetized_waveform, ax=axes[0][0])
        graficar_mel(phonetized_mel, ax=axes[0][1])
        graficar_waveform(spoken_waveform, ax=axes[1][0])
        graficar_mel(spoken_mel, ax=axes[1][1])
        plt.show()

    display(widgets.VBox([out, widgets.HBox([audio_widget(phonetized_waveform), audio_widget(spoken_waveform)])]))

## Speech Autoencoder

### Definición

In [ ]:
from cowaver.modules.reading import SpeechAutoEncoder

speech_autoencoder = SpeechAutoEncoder(h_dim=H_DIM, n_filters=N_FILTERS)
print(speech_autoencoder)

### Entrenamiento

In [ ]:
from cowaver.modules.reading import SpeechAutoEncoder
from cowaver.models import TrainProgramme
from cowaver.utils import sembrar_semilla, entrenar_red
from cowaver.checkpoints import guardar_checkpoint

programme = TrainProgramme(
    theta_max=30,
    epsilon_zero=1e-3,
    theta=20,
    epsilon_theta=1e-4,
    patience=3
)

# Uncomment to train
#
sembrar_semilla(40)
speech_autoencoder = SpeechAutoEncoder(h_dim=H_DIM, n_filters=N_FILTERS)
speech_autoencoder, train_history = entrenar_red(net=speech_autoencoder, data=full_data, programme=programme)
guardar_checkpoint(speech_autoencoder, train_history=train_history, programme=programme, folder=checkpoints_path)

### Evaluación

In [ ]:
from cowaver.modules.reading import SpeechAutoEncoder
from cowaver.utils import evaluar_red
from cowaver.checkpoints import cargar_checkpoint
from cowaver.plots import graficar_train_history, graficar_test_results

speech_autoencoder = SpeechAutoEncoder(h_dim=H_DIM, n_filters=N_FILTERS)
train_history = cargar_checkpoint(speech_autoencoder, folder=checkpoints_path, silent=True)
test_results = evaluar_red(speech_autoencoder, full_data)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
graficar_train_history(train_history, ax=axes[0])
graficar_test_results(test_results, ax=axes[1])
plt.show()

### Exploración

In [ ]:
import matplotlib.pyplot as plt
from cowaver.modules.reading import SpeechAutoEncoder
from cowaver.utils import extraer_waveform
from cowaver.plots import graficar_waveform, graficar_mel

speech_autoencoder = SpeechAutoEncoder(h_dim=H_DIM, n_filters=N_FILTERS)
cargar_checkpoint(speech_autoencoder, folder=checkpoints_path, silent=True)
speech_autoencoder.eval()
exploration_set = full_data.test_set

@widgets.interact(idx=(0, len(exploration_set) - 1))
def explore_test_set(idx=0):
    (image, _, spoken_mel), target = exploration_set[idx]
    spoken_waveform = extraer_waveform(spoken_mel)

    with torch.no_grad():
        reconstructed_mel = speech_autoencoder(spoken_mel.unsqueeze(1).transpose(2, 3))
    reconstructed_mel = reconstructed_mel.squeeze(1).transpose(1, 2)
    reconstructed_waveform = extraer_waveform(reconstructed_mel)

    out = widgets.Output()
    with out:
        fig, axes = plt.subplots(2, 2, figsize=(10, 6), layout="constrained")
        fig.suptitle(exploration_set.classes[target])
        graficar_waveform(spoken_waveform, ax=axes[0][0])
        graficar_mel(spoken_mel, ax=axes[0][1])
        graficar_waveform(reconstructed_waveform, ax=axes[1][0])
        graficar_mel(reconstructed_mel, ax=axes[1][1])
        plt.show()
        
    display(widgets.VBox([
        out, 
        widgets.HBox([audio_widget(spoken_waveform), audio_widget(reconstructed_waveform)])]
    ))

## Red de conciencia fonológica

### Definición

In [ ]:
from cowaver.modules.reading import PhonologicalAwareness

phonological_awareness = PhonologicalAwareness(h_dim=H_DIM, n_filters=N_FILTERS)
print(phonological_awareness)

### Entrenamiento

In [ ]:
from cowaver.modules.reading import PhonologicalAwareness
from cowaver.models import TrainProgramme
from cowaver.utils import sembrar_semilla, entrenar_red
from cowaver.checkpoints import guardar_checkpoint

programme = TrainProgramme(
    theta_max=30,
    epsilon_zero=1e-3,
    theta=20,
    epsilon_theta=1e-4,
    patience=3
)

# Uncomment to train
#
sembrar_semilla(40)
phonological_awareness = PhonologicalAwareness(h_dim=H_DIM, n_filters=N_FILTERS)

speech_autoencoder = SpeechAutoEncoder(h_dim=H_DIM, n_filters=N_FILTERS)
cargar_checkpoint(speech_autoencoder, folder=checkpoints_path, silent=True)
phonological_awareness.load_state_dict(speech_autoencoder.state_dict())

phonological_awareness, train_history = entrenar_red(net=phonological_awareness, data=training_data, programme=programme)
guardar_checkpoint(phonological_awareness, train_history=train_history, programme=programme, folder=checkpoints_path)

### Evaluación

In [ ]:
from cowaver.modules.reading import PhonologicalAwareness
from cowaver.utils import evaluar_red
from cowaver.checkpoints import cargar_checkpoint
from cowaver.plots import graficar_train_history, graficar_test_results

phonological_awareness = PhonologicalAwareness(h_dim=H_DIM, n_filters=N_FILTERS)
train_history = cargar_checkpoint(phonological_awareness, folder=checkpoints_path, silent=True)
training_test_results = evaluar_red(phonological_awareness, training_data)
generalization_test_results = evaluar_red(phonological_awareness, generalization_data)

fig, axes = plt.subplots(1, 3, figsize=(10, 4), layout="constrained")
graficar_train_history(train_history, ax=axes[0])
graficar_test_results(training_test_results, ax=axes[1])
graficar_test_results(generalization_test_results, ax=axes[2])
plt.show()

### Exploración

In [ ]:
import matplotlib.pyplot as plt
from cowaver.modules.reading import PhonologicalAwareness
from cowaver.utils import extraer_waveform
from cowaver.plots import graficar_waveform, graficar_mel, graficar_embedding

phonological_awareness = PhonologicalAwareness(h_dim=H_DIM, n_filters=N_FILTERS)
cargar_checkpoint(phonological_awareness, folder=checkpoints_path, silent=True)
phonological_awareness.eval()
exploration_set = full_data.test_set

@widgets.interact(idx=(0, len(exploration_set) - 1))
def explore_test_set(idx=0):
    (image, phonetized_mel, spoken_mel), target = exploration_set[idx]
    phonetized_waveform = extraer_waveform(phonetized_mel)
    spoken_waveform = extraer_waveform(spoken_mel)

    phonetized_mel = phonetized_mel.unsqueeze(1).transpose(2, 3)
    with torch.no_grad():
        h = phonological_awareness.encoder(phonetized_mel)
        mel_hat = phonological_awareness.decoder(h)
    predicted_mel = mel_hat.squeeze(1).transpose(1, 2)
    predicted_waveform = extraer_waveform(predicted_mel)

    out = widgets.Output()
    with out:
        fig, axes = plt.subplots(3, 2, figsize=(10, 6), layout="constrained")
        fig.suptitle(exploration_set.classes[target])
        graficar_waveform(phonetized_waveform, ax=axes[0][0])
        graficar_mel(phonetized_mel, ax=axes[0][1])
        graficar_waveform(spoken_waveform, ax=axes[1][0])
        graficar_mel(spoken_mel, ax=axes[1][1])
        graficar_embedding(h, ax=axes[2][0], vmin=h.min(), vmax=h.max())
        graficar_mel(predicted_mel, ax=axes[2][1])
        plt.show()
        
    display(widgets.VBox([
        out, 
        widgets.HBox([audio_widget(phonetized_waveform), audio_widget(spoken_waveform), audio_widget(predicted_waveform)])]
    ))

## Ruta fonológica de la lectura

### Definición

In [ ]:
from cowaver.modules.reading import PhonologicalRoute

phonological_route = PhonologicalRoute(h_dim=H_DIM, n_filters=N_FILTERS)
print(phonological_route)

### Entrenamiento

In [ ]:
from cowaver.modules.reading import PhonologicalRoute
from cowaver.models import TrainProgramme
from cowaver.utils import sembrar_semilla, entrenar_red
from cowaver.checkpoints import cargar_checkpoint, guardar_checkpoint

programme = TrainProgramme(
    theta_max=60,
    epsilon_zero=1e-3,
    theta=45,
    epsilon_theta=1e-4,
    patience=7
)

# Uncomment to train
#
sembrar_semilla(42)
phonological_route = PhonologicalRoute(h_dim=H_DIM, n_filters=N_FILTERS)

checkpoint = torch.utils.model_zoo.load_url(CORNET_CKPT_URL, map_location=torch.device("cpu"))
phonological_route.cornet.load_state_dict(checkpoint["state_dict"], strict=False)

cargar_checkpoint(phonological_route.phonological_awareness, folder=checkpoints_path, silent=True)

print("Midiendo estadística", end="... ")
all_h = []
phonological_route.phonological_awareness.eval()
with torch.no_grad():
    for batch in training_data.train_loader():
        (_, phonetized_mel, _), labels = batch
        phonetized_mel = phonetized_mel.transpose(2, 3)
        h = phonological_route.phonological_awareness.encoder(phonetized_mel)
        all_h.append(h)
all_h = torch.cat(all_h, dim=0)
h_mean = all_h.mean(dim=0)
h_std = all_h.std(dim=0)
phonological_route.set_h_stats(h_mean, h_std)
print("OK")

phonological_route, train_history = entrenar_red(net=phonological_route, data=training_data, programme=programme)
guardar_checkpoint(phonological_route, train_history=train_history, programme=programme, folder=checkpoints_path)

### Evaluación

In [ ]:
from cowaver.modules.reading import PhonologicalRoute
from cowaver.checkpoints import cargar_checkpoint
from cowaver.utils import evaluar_red
from cowaver.plots import graficar_train_history, graficar_test_results

phonological_route = PhonologicalRoute(h_dim=H_DIM, n_filters=N_FILTERS)
train_history = cargar_checkpoint(phonological_route, folder=checkpoints_path, silent=True)
training_test_results = evaluar_red(phonological_route, training_data)
generalization_test_results = evaluar_red(phonological_route, generalization_data)

fig, axes = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")
graficar_train_history(train_history, ax=axes[0])
graficar_test_results(training_test_results, ax=axes[1])
graficar_test_results(generalization_test_results, ax=axes[2])
plt.show()

### Exploración

In [ ]:
import matplotlib.pyplot as plt
from cowaver.utils import extraer_waveform, make_image
from cowaver.plots import graficar_waveform, graficar_mel, graficar_embedding

phonological_route = PhonologicalRoute(h_dim=H_DIM, n_filters=N_FILTERS)
cargar_checkpoint(phonological_route, folder=checkpoints_path, silent=True)
phonological_route.eval()

@widgets.interact(word="buzo")
def explore_phonological_route(word: str):
    image = make_image(word)
    with torch.no_grad():
        mel, h = phonological_route(image)
    mel = mel.squeeze(1).transpose(1, 2)
    waveform = extraer_waveform(mel)

    out = widgets.Output()
    with out:
        fig, axes = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")
        fig.suptitle(word)
        graficar_waveform(waveform, ax=axes[0])
        graficar_mel(mel, ax=axes[1])
        graficar_embedding(h, ax=axes[2], vmin=h.min(), vmax=h.max())
        plt.show()
        
    display(widgets.VBox([out, audio_widget(waveform)]))